In [ ]:
class MancalaBoard:
  def __init__(self):

      self.board = [
          [4, 4, 4, 4, 4, 4, 0],
          [4, 4, 4, 4, 4, 4, 0],
      ]
      self.current_player = 0

  def copy(self):
      new_board = MancalaBoard()
      new_board.board = [row[:] for row in self.board]
      new_board.current_player = self.current_player
      return new_board

  def get_legal_moves(self, player):
  # This will return pits that aren't empty (indices 0 to 5)
    return [i for i in range(6) if self.board[player][i] > 0]


  def is_terminal(self):
  # The Game will end when either player's pits will be all empty
    return (all(self.board[0][i] == 0 for i in range(6)) or
          all(self.board[1][i] == 0 for i in range(6)))

  def collect_remaining(self):
      for player in range(2):
          for pit in range(6):
              self.board[player][6] += self.board[player][pit]
              self.board[player][pit] = 0

  def distribute_stones(self, player, pit):
    stones = self.board[player][pit]
    self.board[player][pit] = 0
    current_side = player
    current_pit = pit
    bonus_turn = False

    while stones > 0:
        current_pit += 1

        if current_pit == 6:
            if current_side == player:
                self.board[player][6] += 1
                stones -= 1
                if stones == 0:
                    bonus_turn = True
                    break
                current_side = 1 - current_side
                current_pit = -1
                continue

        if stones == 0 and current_side == player:
            if self.board[player][current_pit] == 1:
                opposite_pit = 5 - current_pit
                captured = self.board[1 - player][opposite_pit]
                if captured > 0:
                    self.board[player][6] += captured + 1
                    self.board[1 - player][opposite_pit] = 0

        return bonus_turn

  def make_move(self, pit):
      player = self.current_player
      bonus_turn = self.distribute_stones(player, pit)
      if not bonus_turn:
          self.current_player = 1 - player
      return self


In [ ]:
game = MancalaBoard()
print(game.get_legal_moves(0))
print(game.is_terminal())

[0, 1, 2, 3, 4, 5]
False


Need to add distrubute_stones and make_move right after it

Note for difficulties had to think about the logic of the setup of the Mancala Board and initially seperated parts but when combined then it works out

After add distrubute_stones and make_move right after it then create the actual algorithm

initial bug space in variable name
wrong variable name for current player
missing return best_value in the maximising block
return is inside the for loop instead of the outside

In [ ]:
#This is the Alpha-Beta Minimax which will be the core of the project
def evaluate(board,player):
  #This will be store the differential H4 from the research
  return board.board[player][6] - board.board[1-player][6]

def alpha_beta(board, depth, alpha, beta, player):
    if board.is_terminal() or depth == 0:
        if board.is_terminal():
            board.collect_remaining()
        return evaluate(board,player)

    current = board.current_player

    if current == player: # This is Maximizing
        best_value = float('-inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)
            # It is important to make sure to not decrement depth on the bonus turns
            new_depth = depth if new_board.current_player == current else depth -1
            value = alpha_beta(new_board, new_depth, alpha, beta, player)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)
            if beta <= alpha:
              break # We will prune now
            return best_value

    else: # This is the minimizing part
        best_value = float('inf')
        for move in board.get_legal_moves(current):
            new_board = board.copy()
            new_board.make_move(move)
            new_depth = depth if new_board.current_player == current else depth -1
            value= alpha_beta(new_board, new_depth, alpha, beta, player)
            best_value = min(best_value, value)
            beta = min(beta, best_value)
            if beta <= alpha:
                  break # Prune
            return best_value

def minimax_agent(board, depth=8):
    player = board.current_player
    best_move = None
    best_value = float('-inf')

    for move in board.get_legal_moves(player):
        new_board = board.copy()
        new_board.make_move(move)
        value = alpha_beta(new_board, depth -1, float('-inf'), float('inf'), player)
        if value > best_value:
              best_value = value
              best_move = move

    return best_move


In [ ]:
game = MancalaBoard()
move = minimax_agent(game, depth=4)
print(f"Best move for Player 1:Pit {move}") # This should print a number 0-5

Best move for Player 1:Pit 5


In [ ]:
def H1(board, player):
    """Hoard seeds in the leftmost pit (furthest from own store)."""
    return board.board[player][0]

def H4(board, player):
    """Number of seeds in own store."""
    return board.board[player][6]

def H6(board, player):
    """Negative of opponent's store; we want this small."""
    return -board.board[1 - player][6]

def H7(board, player):
    """1 if it is still player's turn at this state (last move earned a bonus turn)."""
    return 1 if board.current_player == player else 0

# Weights derived from the average values in Hunter (2021) Table 3.
W1, W4, W6, W7 = 0.2, 1.0, 0.6, 0.9

def evaluate_heuristic(board, player):
    return (W1 * H1(board, player)
          + W4 * H4(board, player)
          + W6 * H6(board, player)
          + W7 * H7(board, player))

In [ ]:
import time

# Heuristic snapshot on the starting board
b = MancalaBoard()
print(f"Start position H1={H1(b,0)}, H4={H4(b,0)}, H6={H6(b,0)}, H7={H7(b,0)}, "
      f"weighted={evaluate_heuristic(b,0):.2f}")

# Both evaluators on a fresh board, depth 6
t0 = time.time()
m1 = minimax_agent(MancalaBoard(), depth=6)
print(f"Plain alpha-beta picks pit {m1} in {time.time()-t0:.2f}s")

t0 = time.time()
m2 = minimax_agent(MancalaBoard(), depth=6, evaluate_fn=evaluate_heuristic)
print(f"Heuristic minimax picks pit {m2} in {time.time()-t0:.2f}s")

# One full game: heuristic (P0) vs plain (P1) at depth 4
g = MancalaBoard()
while not g.is_terminal():
    fn = evaluate_heuristic if g.current_player == 0 else evaluate
    move = minimax_agent(g, depth=4, evaluate_fn=fn)
    if move is None:
        break
    g.make_move(move)
g.collect_remaining()
print(f"Final score - P0 heuristic: {g.board[0][6]}, P1 plain: {g.board[1][6]}")

Start position H1=4, H4=0, H6=0, H7=1, weighted=1.70
Plain alpha-beta picks pit 2 in 0.43s
Heuristic minimax picks pit 5 in 0.71s
Final score - P0 heuristic: 40, P1 plain: 8


In [ ]:
g = MancalaBoard()
print("=== Starting position ===")
print(g)

move_num = 0
while not g.is_terminal():
    fn = evaluate_heuristic if g.current_player == 0 else evaluate
    move = minimax_agent(g, depth=4, evaluate_fn=fn)
    if move is None:
        break
    move_num += 1
    print(f"\n--- Move {move_num}: P{g.current_player} plays pit {move} ---")
    g.make_move(move)
    print(g)

g.collect_remaining()
print(f"\nFinal score: P0 (heuristic) = {g.board[0][6]}, P1 (plain) = {g.board[1][6]}")

=== Starting position ===
    4    4    4    4    4    4   
 0                              0
    4    4    4    4    4    4   

--- Move 1: P0 plays pit 5 ---
    4    4    4    5    5    5   
 0                              1
    4    4    4    4    4    0   

--- Move 2: P1 plays pit 1 ---
    5    5    5    6    0    5   
 1                              1
    4    4    4    4    4    0   

--- Move 3: P1 plays pit 0 ---
    6    6    6    7    1    0   
 1                              1
    4    4    4    4    4    0   

--- Move 4: P0 plays pit 2 ---
    6    6    6    7    1    0   
 1                              2
    4    4    0    5    5    1   

--- Move 5: P0 plays pit 5 ---
    6    6    6    7    1    0   
 1                              3
    4    4    0    5    5    0   

--- Move 6: P0 plays pit 4 ---
    6    6    6    8    2    1   
 1                              4
    4    4    0    5    0    1   

--- Move 7: P1 plays pit 1 ---
    6    6    7    9    0    1   
 1